In [ ]:
import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [ ]:
class RiemannFeaturesDatasetScalerService:
    MAX_LIMIT = 100_000
    def __init__(self, features: list[str]):
        self.__scaler_X = StandardScaler()
        self.__scaler_y = StandardScaler()

        dataset = pd.read_csv("../dataset/riemann_features.csv")

        self.__X_original = dataset.drop(columns=["distance"])
        self.__y_original = dataset["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].to_numpy()[:limit]
        y = self.__y_original.to_numpy()[:limit]

        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        y_train_scaled = self.__scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test_scaled  = self.__scaler_y.transform(y_test.reshape(-1, 1)).ravel()
        
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled

    @property
    def scaler_y(self) -> StandardScaler:
        return self.__scaler_y
    

class JKampeDatasetScalerService:
    MAX_LIMIT = 100_000
    def __init__(self, features: list[str]):
        self.__scaler_X = StandardScaler()
        self.__scaler_y = StandardScaler()
        self.__X_original = pd.read_csv("../dataset/j_kampe.csv")
        self.__y_original = pd.read_csv("../dataset/distances.csv")["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].to_numpy()
        y = self.__y_original.to_numpy().reshape(-1, 1)

        X = X[1_000:limit]
        y = y[1_000:limit]

        X_train = X[:int(0.8 * X.shape[0])]
        X_test  = X[int(0.8 * X.shape[0]):]
        y_train = y[:int(0.8 * y.shape[0])]
        y_test  = y[int(0.8 * y.shape[0]):]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        y_train_scaled = self.__scaler_y.fit_transform(y_train).ravel()
        y_test_scaled  = self.__scaler_y.transform(y_test).ravel()
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled

    @property
    def scaler_y(self) -> StandardScaler:
        return self.__scaler_y